# Configure environment and load data

In [3]:
import os
import pandas as pd
from os.path import join as ospj

qsiprep_dir= "__PROJECT_ROOT__/derivatives/qsiprep"


# Iterate over each subject directory
for group in os.listdir(qsiprep_dir):
    group_path = os.path.join(qsiprep_dir, group)
    if os.path.isdir(group_path):
        group = os.path.basename(group_path)


        dataframes = []
        for sub_dir in os.listdir(group_path):
            sub_path = os.path.join(group_path, sub_dir)
            if os.path.isdir(sub_path):
                sub = os.path.basename(sub_path)
                # Iterate over each session directory

                if group in ["penn_epilepsy", "penn_controls"]:
                    for ses_dir in os.listdir(sub_path):
                        ses_path = os.path.join(sub_path, ses_dir)
                        if os.path.isdir(ses_path):
                            ses = os.path.basename(ses_path)
                            qc_file = os.path.join(ses_path, "dwi", f"{sub}_{ses}_space-ACPC_desc-image_qc.tsv")
                            if os.path.isfile(qc_file):
                                # Load the tsv file and append to the list
                                df = pd.read_csv(qc_file, sep='\t')
                                dataframes.append(df)

                elif group == "hcpaging":
                    qc_file = os.path.join(sub_path, "dwi", f"{sub}_space-ACPC_desc-image_qc.tsv")
                    if os.path.isfile(qc_file):
                        df = pd.read_csv(qc_file, sep='\t')
                        dataframes.append(df)

        # Concatenate all dataframes
        if dataframes:
            df = pd.concat(dataframes, ignore_index=True)

            qc_dir = os.path.join(qsiprep_dir, "qc")
            os.makedirs(qc_dir, exist_ok=True)
            df.to_csv(os.path.join(qc_dir, f"{group}_dwi_qc.csv"), index=False)
